# 08 Regressions

**Owner:** Yair

**What this notebook does:** tests the research question with regressions: how does democracy change around the start of a conflict, is it different for countries that import many weapons, and does it matter who supplied them?

**Reads:** `panel.csv`, `crosswalk_sipri.csv`, `raw/trade-register.csv`, `raw/UcdpPrioConflict_v26_1.xlsx`, `app/site/mapdata.json` (Part F)

**Writes:** `reg_panel.csv`, `reg_eventstudy.csv`, `reg_suppliers.csv`, `reg_results.csv`

The full instructions, background and hints are in **`docs/regression_guide_yair.pdf`** (same part names as here). Work top to bottom.
**Must do:** Part 0, A, C. **Should do:** B, D. **Nice to have:** E, F, G.

This is a main notebook. It must run from top to bottom with Kernel → Restart and Run All.
Golden rule: none of these models proves cause and effect. Write "countries that ... tended to ...", never "arms cause ...".

## Setup

Install the new packages once (in a terminal, from the repo root), then add them to `requirements.txt`:

`.venv/bin/pip install statsmodels linearmodels lifelines scipy`

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS

RAW = "../data/raw/"
CLEAN = "../data/clean/"
pd.set_option("display.max_columns", 50)

In [ ]:
panel = pd.read_csv(CLEAN + "panel.csv")
print(panel.shape)
print(panel.head(3))

---
# Part 0: The analysis panel

**Step 1. Sort and log arms.** Sort `panel` by `iso3` and `year`. Add `log_arms = np.log1p(arms_imports_tiv)`
(TIV is very skewed: many zeros, a few huge values).

In [ ]:
panel = panel.sort_values(["iso3", "year"]).reset_index(drop=True)
panel["log_arms"] = np.log1p(panel["arms_imports_tiv"])


**Step 2. Onsets.** Add a column `onset`: 1 in a year with `conflict_active == 1` where the **two years before had no conflict**
(the same rule as 06, Part B Step 1), otherwise 0. Hint: per country, `shift(1)` and `shift(2)` of `conflict_active`.
Also add `war_onset`: 1 if the onset year has `max_intensity == 2`.

In [ ]:
panel = panel.sort_values(["iso3", "year"]).copy()

by_country = panel.groupby("iso3")["conflict_active"]
previous_year = by_country.shift(1)
two_years_before = by_country.shift(2)

panel["onset"] = (
    panel["conflict_active"].eq(1)
    & previous_year.eq(0)
    & two_years_before.eq(0)
).astype(int)

panel["war_onset"] = (
    panel["onset"].eq(1) & panel["max_intensity"].eq(2)
).astype(int)



**Step 3. Lags and windows.** Add, per country:
`arms_pre5` = mean of `log_arms` in the 5 years before (shift(1) then rolling(5)),
`arms_10` = sum of `arms_imports_tiv` over the last 10 years (rolling(10, min_periods=1)),
`dem_lag1` = `democracy_index` one year earlier, and `post1991` = 1 from 1992 on.
Save the table as `reg_panel.csv`.

In [ ]:
g = panel.groupby("iso3", sort=False)
panel["arms_pre5"] = g["log_arms"].transform(lambda s: s.shift(1).rolling(5).mean())
panel["arms_10"] = g["arms_imports_tiv"].transform(lambda s: s.rolling(10, min_periods=1).sum())
panel["dem_lag1"] = g["democracy_index"].shift(1)
panel["post1991"] = (panel["year"] >= 1992).astype(int)
panel.to_csv(CLEAN + "reg_panel.csv", index=False)

**Checkpoint.** Run this cell. OK means done. Do not change it.

In [ ]:
rp = pd.read_csv(CLEAN + "reg_panel.csv")
for col in ["iso3", "year", "log_arms", "onset", "war_onset", "arms_pre5", "arms_10", "dem_lag1", "post1991"]:
    assert col in rp.columns, "missing column " + col
assert not rp.duplicated(["iso3", "year"]).any()
assert rp["onset"].isin([0, 1]).all()
assert int(rp["onset"].sum()) == 281, "06 found 281 onsets with the same rule; you have " + str(int(rp["onset"].sum()))
assert (rp["log_arms"] >= 0).all()
print("onsets:", int(rp["onset"].sum()), "of which wars:", int(rp["war_onset"].sum()))
print("OK")

---
# Part A: The main finding as a regression (must do)

**Step 1. Outcomes.** For k = 1 to 10 make `y_k = democracy_index(t + k) - democracy_index(t - 1)` per country
(shift(-k) and shift(1)). Standardise `arms_pre5` into `arms_z` (mean 0, sd 1).

In [ ]:
# your code here

**Step 2. A1 regression for each k.** `y_k ~ onset + onset:arms_z + arms_z + EntityEffects + TimeEffects` with `PanelOLS`,
data indexed by `["iso3", "year"]`, `cov_type="clustered", cluster_entity=True`. Collect for every k: `b1` (onset),
`b2` (onset:arms_z), their 95% intervals, `n_obs` and `n_countries` in a dataframe `a1`.

In [ ]:
# your code here

**Checkpoint.** Run this cell. OK means done. Do not change it.

In [ ]:
assert isinstance(a1, pd.DataFrame), "make a dataframe a1"
assert len(a1) == 10, "one row per k = 1 ... 10"
for col in ["k", "b1", "b2", "n_obs"]:
    assert col in a1.columns, "missing column " + col
assert a1["n_obs"].is_monotonic_decreasing or a1["n_obs"].nunique() > 1, "n_obs should shrink as k grows"
print(a1[["k", "b1", "b2", "n_obs"]])
print("OK")

**Step 3. Plot A1.** b1 and b1 + b2 (one sd more arms imports) against k, with 95% bands and a dashed zero line.

In [ ]:
# your code here

**Your interpretation** (2 to 4 sentences, no causal words like "causes" or "leads to"):

...

**Step 4. A2 event study.** Event-time dummies for t = -10 ... +10 around each onset (bin the ends, leave out t = -1 as reference),
outcome `democracy_index`, same fixed effects and clustering. Plot the 21 coefficients with intervals and a dashed line at t = 0.
Are the leads (t = -10 ... -2) close to zero?

In [ ]:
# your code here

**Step 5. A3 stacked event study.** For each onset: the onset country for t = -10 ... +10 plus clean controls (no conflict in that
21-year window), one stack id each. Stack them, use stack-by-country and stack-by-year fixed effects, estimate the event dummies
interacted with "treated". Then do it separately for high and low importers (or with arms_pre5). See the guide, Part A3.
Save `reg_eventstudy.csv` with columns `t, group, coef, ci_low, ci_high, n` (group: "all", "high importer", "low importer").

In [ ]:
# your code here

**Checkpoint.** Run this cell. OK means done. Do not change it.

In [ ]:
es = pd.read_csv(CLEAN + "reg_eventstudy.csv")
assert list(es.columns) == ["t", "group", "coef", "ci_low", "ci_high", "n"]
assert es["t"].min() == -10 and es["t"].max() == 10
assert "all" in set(es["group"])
assert (es["ci_low"] <= es["coef"]).all() and (es["coef"] <= es["ci_high"]).all()
ref = es[es["t"] == -1]
assert len(ref) == 0 or ref["coef"].abs().max() < 1e-9, "t = -1 is the reference: leave it out or set it to 0"
print(es[es["group"] == "all"].head(12))
print("OK")

**Your interpretation** (2 to 4 sentences, no causal words like "causes" or "leads to"):

...

---
# Part B: Does it depend on the kind of case? (should do)

**Step 1. Conflict type at onset.** Read the raw UCDP file as in 02 and attach `type_of_conflict` (1 to 4) of the conflict that
started in each onset year.

In [ ]:
# your code here

**Step 2. Splits.** Run the A1 model for k = 5 and k = 10 with one extra interaction at a time: `war_onset`, conflict type,
`post1991`, starting regime (`regime_type` at t - 1). One table: split, b1, b2, intervals, number of onsets, number of countries.
Groups with fewer than about 15 onsets: report, do not interpret.

In [ ]:
# your code here

**Your interpretation** (2 to 4 sentences, no causal words like "causes" or "leads to"):

...

---
# Part C: Who supplied the arms (must do)

**Step 1. Full supplier table.** Rebuild deliveries per recipient, supplier and year from `trade-register.csv` with the cleaning of 04,
but keep every supplier. Map supplier names to iso3 (crosswalk_sipri plus: Soviet Union → RUS, Czechoslovakia → CZE, Yugoslavia → SRB,
East Germany (GDR) → DDR, United States → USA). Drop non-state suppliers. Print the unmatched names and the TIV they cover.

In [ ]:
# your code here

**Step 2. Supplier democracy.** Merge each supplier's `democracy_index` and `regime_type` in the delivery year. Per recipient and year:
`sup_dem` = TIV-weighted mean supplier democracy, `share_dem` = share of TIV from suppliers with regime_type 2 or 3.
Save `reg_suppliers.csv` with columns `iso3, year, sup_dem, share_dem`.

In [ ]:
# your code here

**Checkpoint.** Run this cell. OK means done. Do not change it.

In [ ]:
sp = pd.read_csv(CLEAN + "reg_suppliers.csv")
assert list(sp.columns) == ["iso3", "year", "sup_dem", "share_dem"]
assert sp["sup_dem"].between(0, 1).all(), "sup_dem is a democracy index, 0 to 1"
assert sp["share_dem"].between(0, 1).all(), "share_dem is a share, 0 to 1"
assert not sp.duplicated(["iso3", "year"]).any()
print(sp.groupby(sp["year"] < 1992)["sup_dem"].mean())
print("OK")

**Step 3. Regression.** Make `sup_dem_pre5` (5-year pre-window mean) and add `onset:sup_dem_pre5` to the A1 model for k = 5 and 10,
keeping `onset:arms_z`. Only countries with imports in the window.

In [ ]:
# your code here

**Your interpretation** (2 to 4 sentences, no causal words like "causes" or "leads to"):

...

**Step 4 (optional). Supplier switching.** Main supplier = most TIV in a 5-year window; switch = it changes. Does a regime change raise
the chance of a switch in the next 5 years? Linear probability model with country and year fixed effects.

In [ ]:
# your code here

---
# Part D: Turn the question around (should do)

**Step 1. What predicts an onset?** Sample: country-years with no conflict in the two previous years.
`onset ~ dem_lag1 + dem_lag1^2 + log_arms(t-1) + regime dummies(t-1)`: (a) linear probability model with country and year fixed effects,
(b) `smf.logit` with year dummies, report average marginal effects. Plot the predicted onset probability over democracy 0 to 1.

In [ ]:
# your code here

**Step 2. Arms before or after?** Run the Part A event study with `log_arms` as the outcome. Same chart format.

In [ ]:
# your code here

**Your interpretation** (2 to 4 sentences, no causal words like "causes" or "leads to"):

...

---
# Part E: Survival models (nice to have)

**Step 1. Recovery time.** Per onset: years until democracy is back at its t = -1 level (censored at 2025 or the next onset).
Kaplan-Meier curves for high vs low importers, then a Cox model with arms_pre5, war_onset, regime at t - 1 and post1991 (lifelines).

In [ ]:
# your code here

**Step 2. Risk of autocratisation.** Discrete-time hazard: one row per country-year, event = regime_type drops by one or more.
Logit on conflict_active, log_arms(t-1), dem_lag1 and years-since-last-change dummies.

In [ ]:
# your code here

---
# Part F: Neighbours (nice to have)

**Step 1. Neighbour list** from `../app/site/mapdata.json` (Equal Earth map, 960 x 423): two countries touch if about 10 of their
points are within 1 map unit. Check Germany-France (yes) and Germany-Spain (no).

In [ ]:
# your code here

**Step 2. Spillover.** Share of neighbours with conflict in t - 1 and mean neighbour log_arms; regress own log_arms and the change in
democracy on them (country and year fixed effects).

In [ ]:
# your code here

---
# Part G: GDP and population controls (nice to have)

Download the Maddison Project Database 2023 into `data/raw/`, record it in CONVENTIONS.md, merge on iso3 and year, and add log GDP per
capita and log population (t - 1) to the A1, C3 and D1 models. Do the main results keep sign and size?

In [ ]:
# your code here

---
# Results file and findings

**Step 1. Results table.** One row per model and term: `model, term, coef, se, ci_low, ci_high, n_obs, n_countries`.
Save it as `reg_results.csv`. Then add the four new files to CONVENTIONS.md (made by 08).

**Model names the website reads** (use exactly these in the `model` and `term` columns):

| Part | model | term(s) |
| --- | --- | --- |
| A1 | `A1_k1` ... `A1_k10` | `onset`, `onset:arms_z` |
| B | `B_war_k5`, `B_type_k5`, `B_post1991_k5`, `B_regime_k5` | the interaction, e.g. `onset:war_onset`, `onset:intrastate`, `onset:post1991`, `onset:democracy_t1` |
| C3 | `C3_k5`, `C3_k10` | `onset:sup_dem_pre5` |
| D1 | `D1_lpm` | `dem_lag1`, `dem_lag1_sq`, `log_arms_lag1` |

Other models can be added with any name; the website lists them in a general table.
Until your files exist in `data/clean/`, the website shows placeholder numbers from `data/demo/` with a warning banner.

In [ ]:
# your code here

**Checkpoint.** Run this cell. OK means done. Do not change it.

In [ ]:
rr = pd.read_csv(CLEAN + "reg_results.csv")
assert list(rr.columns) == ["model", "term", "coef", "se", "ci_low", "ci_high", "n_obs", "n_countries"]
assert len(rr) > 0 and rr["se"].ge(0).all()
print(rr["model"].unique())
print("OK")

## Findings (write in your own words)

1. Main result (Part A):
2. Does arms importing matter (A, B)?
3. Does the supplier matter (C)?
4. What the reverse checks show (D):
5. Limits: endogeneity, small groups, historical states, TIV is not money.

Then tell Nicolas that `reg_eventstudy.csv` and `reg_suppliers.csv` exist, for the website.